# Klasyfikacja - K-Nearest Neighbors

### Analiza modelu:
1. [Import bibliotek](#0)
2. [Wprowadzenie do modelu](#1)
3. [Strategia balansowania](#2)
4. [Analiza dla datasetu Mushroom](#3)
5. [Analiza dla datasetu Adult Income](#4)
6. [Porownanie wynikow](#5)
7. [Wnioski koncowe](#6)


### <a name='0'></a> Import bibliotek

W notebooku wykorzystujemy wspolny pipeline przygotowania danych,
aby porownanie tego samego algorytmu na dwoch datasetach bylo uczciwe.


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from mushroom_analysis import (
    TARGET_COLUMN,
    BernoulliNB,
    DecisionTreeClassifier,
    KNeighborsClassifier,
    LogisticRegression,
    RANDOM_STATE,
    RandomForestClassifier,
    class_balance_table,
    evaluate_models,
    load_data,
    load_dataset,
    missing_values_table,
    oversample_minority_class,
    plot_class_distribution,
    plot_confusion_matrix_for_model,
    plot_feature_histograms,
    plot_top_feature_correlations,
    preprocess_after_split,
    split_dataset,
)


### <a name='1'></a> Wprowadzenie do modelu


K-Nearest Neighbors to metoda oparta na podobienstwie obserwacji.
Nowy przypadek klasyfikowany jest na podstawie etykiet jego najblizszych sasiadow
w przestrzeni cech. Po zastosowaniu One-Hot Encoding model ten moze skutecznie
porownywac obserwacje zapisane w postaci wektorow binarnych.


### <a name='2'></a> Strategia balansowania


Dla modelu KNN zastosowano oversampling klasy mniejszosciowej wyłącznie
na zbiorze treningowym. Podejscie to ma sens, poniewaz KNN nie korzysta
z parametru `class_weight`, a liczba sasiadow moze byc wrazliwa
na nierownowage klas.


### <a name='3'></a> Analiza dla datasetu Mushroom

Najpierw uruchamiamy ten sam algorytm na zbiorze mushroom.
Pozwoli to sprawdzic, czy model nadal osiaga niemal idealne wyniki
na bardzo dobrze separowalnym problemie klasyfikacyjnym.


In [ ]:
df = load_dataset("mushroom")
train_df, validation_df, test_df = split_dataset(df)
preprocessed = preprocess_after_split(train_df, validation_df, test_df)

X_train_reference, y_train_reference = oversample_minority_class(
    preprocessed['X_train_model'],
    preprocessed["y_train"],
)
X_validation_reference = preprocessed['X_validation_model']
X_test_reference = preprocessed['X_test_model']

print("Dataset:", "mushroom")
print("Shape:", df.shape)
print(pd.Series(y_train_reference).value_counts())


#### Definicja modelu


In [ ]:
model = KNeighborsClassifier(n_neighbors=11)
model


In [ ]:
model.fit(X_train_reference, y_train_reference)

validation_pred = model.predict(X_validation_reference)
test_pred = model.predict(X_test_reference)

metrics_df = pd.DataFrame(
    [
        {
            "split": "validation",
            "accuracy": round(accuracy_score(preprocessed["y_validation"], validation_pred), 4),
            "precision": round(precision_score(preprocessed["y_validation"], validation_pred), 4),
            "recall": round(recall_score(preprocessed["y_validation"], validation_pred), 4),
            "f1": round(f1_score(preprocessed["y_validation"], validation_pred), 4),
        },
        {
            "split": "test",
            "accuracy": round(accuracy_score(preprocessed["y_test"], test_pred), 4),
            "precision": round(precision_score(preprocessed["y_test"], test_pred), 4),
            "recall": round(recall_score(preprocessed["y_test"], test_pred), 4),
            "f1": round(f1_score(preprocessed["y_test"], test_pred), 4),
        },
    ]
)
metrics_df


In [ ]:
mushroom_metrics = metrics_df.copy()
mushroom_metrics


In [ ]:
validation_row = metrics_df[metrics_df["split"] == "validation"].iloc[0]
test_row = metrics_df[metrics_df["split"] == "test"].iloc[0]

print(
    f"Wnioski dla mushroom: validation F1 = {validation_row['f1']:.4f}, "
    f"test F1 = {test_row['f1']:.4f}, validation accuracy = {validation_row['accuracy']:.4f}, "
    f"test accuracy = {test_row['accuracy']:.4f}."
)


#### Macierz pomylek i raport klasyfikacji


In [ ]:
mushroom_classification_report = plot_confusion_matrix_for_model(
    model,
    X_test_reference,
    preprocessed["y_test"],
    preprocessed["label_encoder"],
)
mushroom_classification_report


### <a name='4'></a> Analiza dla datasetu Adult Income

Nastepnie uruchamiamy ten sam algorytm na bardziej wymagajacym
zewnetrznym datasecie Adult Income. Ten zbior zawiera cechy mieszane
i jest znacznie mniej separowalny niz mushroom.


In [ ]:
df = load_dataset("adult_income")
train_df, validation_df, test_df = split_dataset(df)
preprocessed = preprocess_after_split(train_df, validation_df, test_df)

X_train_reference, y_train_reference = oversample_minority_class(
    preprocessed['X_train_model'],
    preprocessed["y_train"],
)
X_validation_reference = preprocessed['X_validation_model']
X_test_reference = preprocessed['X_test_model']

print("Dataset:", "adult_income")
print("Shape:", df.shape)
print(pd.Series(y_train_reference).value_counts())


#### Definicja modelu


In [ ]:
model = KNeighborsClassifier(n_neighbors=11)
model


In [ ]:
model.fit(X_train_reference, y_train_reference)

validation_pred = model.predict(X_validation_reference)
test_pred = model.predict(X_test_reference)

metrics_df = pd.DataFrame(
    [
        {
            "split": "validation",
            "accuracy": round(accuracy_score(preprocessed["y_validation"], validation_pred), 4),
            "precision": round(precision_score(preprocessed["y_validation"], validation_pred), 4),
            "recall": round(recall_score(preprocessed["y_validation"], validation_pred), 4),
            "f1": round(f1_score(preprocessed["y_validation"], validation_pred), 4),
        },
        {
            "split": "test",
            "accuracy": round(accuracy_score(preprocessed["y_test"], test_pred), 4),
            "precision": round(precision_score(preprocessed["y_test"], test_pred), 4),
            "recall": round(recall_score(preprocessed["y_test"], test_pred), 4),
            "f1": round(f1_score(preprocessed["y_test"], test_pred), 4),
        },
    ]
)
metrics_df


In [ ]:
adult_metrics = metrics_df.copy()
adult_metrics


In [ ]:
validation_row = metrics_df[metrics_df["split"] == "validation"].iloc[0]
test_row = metrics_df[metrics_df["split"] == "test"].iloc[0]

print(
    f"Wnioski dla adult_income: validation F1 = {validation_row['f1']:.4f}, "
    f"test F1 = {test_row['f1']:.4f}, validation accuracy = {validation_row['accuracy']:.4f}, "
    f"test accuracy = {test_row['accuracy']:.4f}."
)


#### Macierz pomylek i raport klasyfikacji


In [ ]:
adult_classification_report = plot_confusion_matrix_for_model(
    model,
    X_test_reference,
    preprocessed["y_test"],
    preprocessed["label_encoder"],
)
adult_classification_report


### <a name='5'></a> Porownanie wynikow

W tej sekcji zestawiamy wyniki jednego algorytmu
na dwoch roznych datasetach.


In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            "dataset": "mushroom",
            "validation_accuracy": mushroom_metrics[mushroom_metrics["split"] == "validation"].iloc[0]["accuracy"],
            "validation_f1": mushroom_metrics[mushroom_metrics["split"] == "validation"].iloc[0]["f1"],
            "test_accuracy": mushroom_metrics[mushroom_metrics["split"] == "test"].iloc[0]["accuracy"],
            "test_f1": mushroom_metrics[mushroom_metrics["split"] == "test"].iloc[0]["f1"],
        },
        {
            "dataset": "adult_income",
            "validation_accuracy": adult_metrics[adult_metrics["split"] == "validation"].iloc[0]["accuracy"],
            "validation_f1": adult_metrics[adult_metrics["split"] == "validation"].iloc[0]["f1"],
            "test_accuracy": adult_metrics[adult_metrics["split"] == "test"].iloc[0]["accuracy"],
            "test_f1": adult_metrics[adult_metrics["split"] == "test"].iloc[0]["f1"],
        },
    ]
)
comparison_df


In [ ]:
mushroom_row = comparison_df[comparison_df["dataset"] == "mushroom"].iloc[0]
adult_row = comparison_df[comparison_df["dataset"] == "adult_income"].iloc[0]

print(
    f"Wnioski koncowe dla K-Nearest Neighbors: model osiagnal na mushroom test F1 = "
    f"{mushroom_row['test_f1']:.4f}, natomiast na adult_income test F1 = "
    f"{adult_row['test_f1']:.4f}. Oznacza to, ze mushroom jest datasetem "
    f"znacznie latwiejszym, a adult_income stanowi bardziej wymagajacy problem klasyfikacyjny."
)


### <a name='6'></a> Wnioski koncowe


KNN rowniez osiagnal bardzo wysokie wyniki, co oznacza,
ze obiekty nalezace do tych samych klas tworza w przestrzeni cech
wyrazne skupienia. Zastosowanie oversamplingu pomoglo zachowac
rownowage podczas klasyfikacji opartej na sasiedztwie.
